# Notebook 06 — Advanced Analyses

This notebook covers three advanced topics:
1. **Ensemble Models** — Voting and Stacking regressors vs. base models
2. **Spatial Analysis** — Geographic choropleth of temperature across locations
3. **Climate Analysis** — Long-term temperature trends by region / continent

Run notebooks 01–05 first (or at minimum 01 to generate cleaned data).

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    VotingRegressor, StackingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
from pathlib import Path

print('Libraries loaded.')

## 1  Load Cleaned Data

In [ ]:
PROCESSED = Path('../data/processed')
CLEANED   = Path('../data/cleaned')

# Try anomaly-filtered first, fall back to cleaned
candidates = [
    PROCESSED / 'weather_no_anomalies.csv',
    PROCESSED / 'weather_cleaned.csv',
    CLEANED   / 'weather_cleaned.csv',
]
df = None
for p in candidates:
    if p.exists():
        df = pd.read_csv(p)
        print(f'Loaded: {p}  ({df.shape})')
        break

if df is None:
    raise FileNotFoundError('Run notebook 01 first to generate cleaned data.')

# Ensure datetime
if 'last_updated' in df.columns:
    df['last_updated'] = pd.to_datetime(df['last_updated'], errors='coerce')

df.head(3)

## 2  Ensemble Models

We compare five models:
- Linear Regression (baseline)
- Random Forest
- Gradient Boosting
- **Voting Ensemble** (average of LR + RF + GB)
- **Stacking Ensemble** (LR + RF + GB meta-learner = Ridge)

In [ ]:
FEATURES = [
    'humidity', 'wind_kph', 'pressure_mb', 'cloud', 'feels_like_celsius',
    'visibility_km', 'uv_index', 'gust_kph',
]
TARGET = 'temperature_celsius'

available = [c for c in FEATURES if c in df.columns]
print('Features available:', available)

sub = df[[TARGET] + available].dropna()
X = sub[available]
y = sub[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
def eval_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    return {
        'Model': name,
        'MAE':   round(mean_absolute_error(y_te, preds), 4),
        'RMSE':  round(mean_squared_error(y_te, preds) ** 0.5, 4),
        'R²':    round(r2_score(y_te, preds), 6),
    }, model

base_estimators = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
]

model_specs = [
    ('Linear Regression',    LinearRegression()),
    ('Random Forest',        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('Gradient Boosting',    GradientBoostingRegressor(n_estimators=200, random_state=42)),
    ('Voting Ensemble',      VotingRegressor(estimators=base_estimators, n_jobs=-1)),
    ('Stacking Ensemble',    StackingRegressor(
                                estimators=base_estimators,
                                final_estimator=Ridge(),
                                cv=5, n_jobs=-1)),
]

results = []
trained = {}
for name, model in model_specs:
    print(f'  Training {name}...')
    row, fitted = eval_model(name, model, X_train, y_train, X_test, y_test)
    results.append(row)
    trained[name] = fitted

metrics_df = pd.DataFrame(results).set_index('Model')
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#4f86c6', '#4caf7d', '#e07b3a', '#9c5fc4', '#e84c4c']

for ax, col in zip(axes, ['MAE', 'RMSE', 'R²']):
    vals = metrics_df[col]
    bars = ax.barh(vals.index, vals.values, color=colors)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Model Comparison — Base vs Ensemble', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/ensemble_comparison.png')

In [ ]:
# Save the best ensemble model
best_name = metrics_df['MAE'].idxmin()
print(f'Best model by MAE: {best_name}')

out_dir = Path('../models')
out_dir.mkdir(exist_ok=True)
joblib.dump(trained[best_name], out_dir / 'best_ensemble.joblib')
print(f'Saved to models/best_ensemble.joblib')

metrics_df.to_csv('../reports/ensemble_metrics.csv')
print('Saved to reports/ensemble_metrics.csv')

## 3  Spatial Analysis

Visualise mean temperature, humidity, and wind speed by geographic location using interactive Plotly maps.

In [ ]:
LAT_COLS = ['latitude', 'lat']
LON_COLS = ['longitude', 'lon', 'long']

lat_col = next((c for c in LAT_COLS if c in df.columns), None)
lon_col = next((c for c in LON_COLS if c in df.columns), None)

if lat_col is None or lon_col is None:
    print('No lat/lon columns found — skipping spatial analysis.')
    print('Available columns:', df.columns.tolist())
else:
    geo = df[[lat_col, lon_col, TARGET] + 
              [c for c in ['humidity', 'wind_kph', 'country'] if c in df.columns]].dropna()
    print(f'Geo records: {len(geo):,}')
    geo.head(3)

In [ ]:
if lat_col and lon_col:
    # Aggregate by location (round to 1 decimal to bin nearby stations)
    geo['lat_r'] = geo[lat_col].round(1)
    geo['lon_r'] = geo[lon_col].round(1)
    agg = geo.groupby(['lat_r', 'lon_r']).agg(
        mean_temp=(TARGET, 'mean'),
        mean_humidity=('humidity', 'mean') if 'humidity' in geo.columns else (TARGET, 'count'),
        count=(TARGET, 'count'),
    ).reset_index()

    fig = px.scatter_geo(
        agg,
        lat='lat_r',
        lon='lon_r',
        color='mean_temp',
        size='count',
        color_continuous_scale='RdBu_r',
        projection='natural earth',
        title='Mean Temperature by Location (°C)',
        labels={'mean_temp': 'Avg Temp (°C)', 'count': 'Observations'},
        size_max=20,
    )
    fig.update_layout(height=500, coloraxis_colorbar=dict(title='°C'))
    fig.show()
    fig.write_html('../reports/spatial_temperature_map.html')
    print('Saved reports/spatial_temperature_map.html')

In [ ]:
if lat_col and lon_col and 'wind_kph' in df.columns:
    wind_agg = geo.groupby(['lat_r', 'lon_r']).agg(
        mean_wind=('wind_kph', 'mean'),
        count=(TARGET, 'count'),
    ).reset_index()

    fig2 = px.scatter_geo(
        wind_agg,
        lat='lat_r',
        lon='lon_r',
        color='mean_wind',
        size='count',
        color_continuous_scale='Blues',
        projection='natural earth',
        title='Mean Wind Speed by Location (kph)',
        labels={'mean_wind': 'Avg Wind (kph)'},
        size_max=20,
    )
    fig2.update_layout(height=500)
    fig2.show()
    fig2.write_html('../reports/spatial_wind_map.html')
    print('Saved reports/spatial_wind_map.html')

In [ ]:
# Country-level choropleth (requires 'country' column with ISO codes or names)
if 'country' in df.columns and lat_col:
    country_agg = df.groupby('country')[TARGET].mean().reset_index()
    country_agg.columns = ['country', 'mean_temp']

    fig3 = px.choropleth(
        country_agg,
        locations='country',
        locationmode='country names',
        color='mean_temp',
        color_continuous_scale='RdBu_r',
        title='Mean Temperature by Country (°C)',
        labels={'mean_temp': 'Avg Temp (°C)'},
    )
    fig3.update_layout(height=500)
    fig3.show()
    fig3.write_html('../reports/choropleth_temperature.html')
    print('Saved reports/choropleth_temperature.html')
else:
    print('No country column — skipping choropleth.')

## 4  Climate Analysis

Examine long-term temperature trends grouped by continent / latitude band and by month.

In [ ]:
def assign_continent(row):
    """Rough latitude-based continental band."""
    lat = row.get(lat_col, 0) if lat_col else 0
    if lat >= 60:    return 'Arctic / N. Europe'
    elif lat >= 35:  return 'Temperate North'
    elif lat >= 15:  return 'Subtropical North'
    elif lat >= -15: return 'Tropics'
    elif lat >= -35: return 'Subtropical South'
    else:            return 'Temperate South'

if lat_col:
    df['climate_zone'] = df.apply(assign_continent, axis=1)
elif 'country' in df.columns:
    # fallback: use country as proxy
    df['climate_zone'] = df['country']
else:
    df['climate_zone'] = 'Unknown'

print(df['climate_zone'].value_counts())

In [ ]:
zone_stats = (
    df.groupby('climate_zone')[TARGET]
    .agg(['mean', 'median', 'std', 'count'])
    .round(2)
    .sort_values('mean')
)
zone_stats.columns = ['Mean (°C)', 'Median (°C)', 'Std (°C)', 'Count']
print(zone_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Box plot by climate zone
zone_order = zone_stats.index.tolist()
data_by_zone = [df[df['climate_zone'] == z][TARGET].dropna().values for z in zone_order]
bp = axes[0].boxplot(data_by_zone, labels=zone_order, patch_artist=True, vert=True)
colors_zone = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(zone_order)))
for patch, color in zip(bp['boxes'], colors_zone):
    patch.set_facecolor(color)
axes[0].set_title('Temperature Distribution by Climate Zone', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].tick_params(axis='x', rotation=30)

# Bar chart of mean temperature
means = zone_stats['Mean (°C)']
bar_colors = ['#e84c4c' if v > 20 else '#4f86c6' for v in means]
axes[1].barh(means.index, means.values, color=bar_colors)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Mean Temperature by Climate Zone', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mean Temperature (°C)')
for i, val in enumerate(means.values):
    axes[1].text(val + 0.3, i, f'{val:.1f}°C', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/climate_zone_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/climate_zone_analysis.png')

In [ ]:
# Monthly temperature trends (if datetime column available)
if 'last_updated' in df.columns and pd.api.types.is_datetime64_any_dtype(df['last_updated']):
    df['month'] = df['last_updated'].dt.month
    df['month_name'] = df['last_updated'].dt.strftime('%b')

    monthly = (
        df.groupby(['month', 'month_name'])[TARGET]
        .agg(['mean', 'std'])
        .reset_index()
        .sort_values('month')
    )

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(monthly['month_name'], monthly['mean'], marker='o', color='#e07b3a',
            linewidth=2, markersize=7, label='Mean Temp')
    ax.fill_between(
        monthly['month_name'],
        monthly['mean'] - monthly['std'],
        monthly['mean'] + monthly['std'],
        alpha=0.2, color='#e07b3a', label='±1 Std'
    )
    ax.set_title('Monthly Temperature Trend (Global)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Temperature (°C)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/monthly_temperature_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved reports/monthly_temperature_trend.png')
else:
    print('No datetime column — skipping monthly trend.')

In [ ]:
# Monthly x Climate Zone heatmap
if 'month' in df.columns:
    pivot = df.pivot_table(
        values=TARGET,
        index='climate_zone',
        columns='month',
        aggfunc='mean',
    )
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]

    fig = px.imshow(
        pivot,
        color_continuous_scale='RdBu_r',
        title='Mean Temperature Heatmap — Climate Zone × Month',
        labels={'color': 'Temp (°C)'},
        aspect='auto',
    )
    fig.update_layout(height=400)
    fig.show()
    fig.write_html('../reports/climate_zone_heatmap.html')
    print('Saved reports/climate_zone_heatmap.html')
else:
    print('No month column — skipping heatmap.')

## 5  Summary

| Analysis | Output |
|----------|--------|
| Ensemble model comparison | `reports/ensemble_metrics.csv`, `reports/ensemble_comparison.png` |
| Best ensemble model saved | `models/best_ensemble.joblib` |
| Spatial temperature map   | `reports/spatial_temperature_map.html` |
| Spatial wind map          | `reports/spatial_wind_map.html` |
| Country choropleth        | `reports/choropleth_temperature.html` |
| Climate zone boxplot      | `reports/climate_zone_analysis.png` |
| Monthly trend             | `reports/monthly_temperature_trend.png` |
| Zone × Month heatmap      | `reports/climate_zone_heatmap.html` |